Codigo final; proceso de conteo.
-Imagen Tornillos

In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
np.seterr(over='raise')


imagen = cv2.imread("img/tornillos_2.jpg", cv2.IMREAD_GRAYSCALE)
_, imagen_bin = cv2.threshold(imagen, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)
cv2.imshow("Bin", imagen_bin)
cv2.waitKey(0)
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(imagen_bin, 8, ltype=cv2.CV_32S)
print(f"Etiquetas_n: {num_labels} Etiquetas: {labels} Estadísticos: {stats} Centroides: {centroids}")

Etiquetas_n: 1517 Etiquetas: [[1 1 1 ... 0 0 0]
 [1 1 1 ... 0 0 0]
 [1 1 1 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]] Estadísticos: [[     0      0    784    485 298260]
 [     0      0      7      5     22]
 [     0      0     47     33    597]
 ...
 [   575    484      2      1      2]
 [   594    484      1      1      1]
 [   607    484      1      1      1]] Centroides: [[419.57320123 240.43485885]
 [  2.09090909   1.40909091]
 [ 17.52428811  12.33333333]
 ...
 [575.5        484.        ]
 [594.         484.        ]
 [607.         484.        ]]


In [4]:
for i in range(1, num_labels):
    x = stats[i, cv2.CC_STAT_LEFT]
    y = stats[i, cv2.CC_STAT_TOP]
    w = stats[i, cv2.CC_STAT_WIDTH]
    h = stats[i, cv2.CC_STAT_HEIGHT]
    area = stats[i, cv2.CC_STAT_AREA]
    (cX, cY) = centroids[i]
    if area > 100:
        cv2.rectangle(imagen, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.circle(imagen, (int(cX), int(cY)), 4, (0, 0, 255), -1)
cv2.imshow('ImagenEtiquetada', imagen)
cv2.waitKey(0)
cv2.destroyAllWindows()


Ajuste de brillo a la imagen

In [5]:
def contraste_brillo(imagen, contraste, brillo):
    imagen_procesada = np.zeros_like(imagen)
    h,w = imagen.shape
    for y in range(h):
        for x in range(w):
            try:
                imagen_procesada[y,x] = np.clip(contraste * imagen[y,x] + brillo, 0, 255).astype(np.uint8)
            except FloatingPointError as e:
                imagen_procesada[y,x] = 255
    return imagen_procesada

ruta = "img/tornillos_2.jpg"
imagen = cv2.imread(ruta, cv2.IMREAD_GRAYSCALE)
h,w = imagen.shape
imagen_procesada = contraste_brillo(imagen, 1, 90)
imagenes_comparadas = np.hstack((imagen, imagen_procesada))
cv2.imshow("Comparacion", imagenes_comparadas)
cv2.waitKey(0)

-1

In [6]:
imagen = cv2.imread("img/tornillos_2.jpg", cv2.IMREAD_GRAYSCALE)
_, imagen_bin = cv2.threshold(imagen_procesada, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)
cv2.imshow("Bin", imagen_bin)
cv2.waitKey(0)
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(imagen_bin, 8, ltype=cv2.CV_32S)
print(f"Etiquetas_n: {num_labels} Etiquetas: {labels} Estadísticos: {stats} Centroides: {centroids}")

Etiquetas_n: 703 Etiquetas: [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]] Estadísticos: [[     0      0    784    485 329918]
 [    33      0      7      5     23]
 [    21      5      4      4      9]
 ...
 [   554    472      1      1      1]
 [   698    475      3      2      4]
 [   551    478      1      1      1]] Centroides: [[401.24172976 242.1186416 ]
 [ 35.82608696   1.47826087]
 [ 22.88888889   6.33333333]
 ...
 [554.         472.        ]
 [699.         475.5       ]
 [551.         478.        ]]


In [7]:
for i in range(1, num_labels):
    x = stats[i, cv2.CC_STAT_LEFT]
    y = stats[i, cv2.CC_STAT_TOP]
    w = stats[i, cv2.CC_STAT_WIDTH]
    h = stats[i, cv2.CC_STAT_HEIGHT]
    area = stats[i, cv2.CC_STAT_AREA]
    (cX, cY) = centroids[i]
    if area > 100:
        cv2.rectangle(imagen_procesada, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.circle(imagen_procesada, (int(cX), int(cY)), 4, (0, 0, 255), -1)
cv2.imshow('ImagenEtiquetada', imagen_procesada)
cv2.waitKey(0)
cv2.destroyAllWindows()

Ajuste de contraste

In [18]:
imagen_color = cv2.imread(ruta)

def contraste_brillo_mat(imagen, contraste, brillo):
    imagen_p = imagen.astype(np.float32)
    return np.clip(contraste * imagen_p + brillo, 0, 255).astype(np.uint8)

imagen_color_HSV = cv2.cvtColor(imagen_color, cv2.COLOR_BGR2HSV)
imagen_v = imagen_color_HSV[:,:,2]
imagen_color_HSV_procesada = contraste_brillo_mat(imagen_v, 1, -120)
imagen_color_HSV[:,:,2] = imagen_color_HSV_procesada
imagen_color_procesada = cv2.cvtColor(imagen_color_HSV, cv2.COLOR_HSV2BGR)
imagen_final_gamma = imagen_color_procesada
cv2.imshow("Imagen final", imagen_final_gamma)
cv2.waitKey(0)

-1

In [21]:
imagen_gray = cv2.cvtColor(imagen_final_gamma, cv2.COLOR_BGR2GRAY)
_, imagen_bin = cv2.threshold(imagen_gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
cv2.imshow("Bin", imagen_bin)
cv2.waitKey(0)
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(imagen_bin, 8, ltype=cv2.CV_32S)
print(f"Etiquetas_n: {num_labels} Etiquetas: {labels} Estadísticos: {stats} Centroides: {centroids}")

Etiquetas_n: 1574 Etiquetas: [[1 1 1 ... 1 1 1]
 [1 1 1 ... 1 1 1]
 [1 1 0 ... 1 1 1]
 ...
 [1 1 1 ... 1 1 1]
 [1 1 1 ... 1 1 1]
 [1 1 1 ... 1 1 1]] Estadísticos: [[     2      2    781    482 114187]
 [     0      0    784    485 262222]
 [     8      3      3      2      4]
 ...
 [   568    482      1      1      1]
 [   608    482      1      1      1]
 [   637    482      2      1      2]] Centroides: [[267.1451216  253.82603098]
 [448.04243732 236.12854375]
 [  9.           3.75      ]
 ...
 [568.         482.        ]
 [608.         482.        ]
 [637.5        482.        ]]


In [22]:
for i in range(1, num_labels):
    x = stats[i, cv2.CC_STAT_LEFT]
    y = stats[i, cv2.CC_STAT_TOP]
    w = stats[i, cv2.CC_STAT_WIDTH]
    h = stats[i, cv2.CC_STAT_HEIGHT]
    area = stats[i, cv2.CC_STAT_AREA]
    (cX, cY) = centroids[i]
    if area > 100:
        cv2.rectangle(imagen_color_procesada, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.circle(imagen_color_procesada, (int(cX), int(cY)), 4, (0, 0, 255), -1)
cv2.imshow('ImagenEtiquetada', imagen_color_procesada)
cv2.waitKey(0)
cv2.destroyAllWindows()

Ajuste de Gamma

In [28]:
imagen_color_HSV = cv2.cvtColor(cv2.imread(ruta), cv2.COLOR_BGR2HSV)
imagen_color_HSV.shape
imagen_v = imagen_color_HSV[:,:,1]
imagen_color_HSV_procesada = contraste_brillo_mat(imagen_v, 1, 90)
imagen_color_HSV[:,:,1] = imagen_color_HSV_procesada
imagen_color_procesada = cv2.cvtColor(imagen_color_HSV, cv2.COLOR_HSV2BGR)
imagen_final_gamma2 = imagen_color_procesada
imagen_mostrar = np.hstack((imagen_color, imagen_color_procesada))
cv2.imshow("Imagenes", imagen_mostrar)
cv2.waitKey(0)

-1

In [29]:
imagen_gray = cv2.cvtColor(imagen_final_gamma2, cv2.COLOR_BGR2GRAY)
_, imagen_bin = cv2.threshold(imagen_gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
cv2.imshow("Bin", imagen_bin)
cv2.waitKey(0)
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(imagen_bin, 8, ltype=cv2.CV_32S)
print(f"Etiquetas_n: {num_labels} Etiquetas: {labels} Estadísticos: {stats} Centroides: {centroids}")

Etiquetas_n: 958 Etiquetas: [[1 0 0 ... 2 2 2]
 [0 0 0 ... 2 2 2]
 [0 0 2 ... 2 2 2]
 ...
 [2 2 2 ... 2 2 2]
 [2 2 2 ... 2 2 2]
 [2 2 2 ... 2 2 2]] Estadísticos: [[     0      0    784    485  83787]
 [     0      0      1      1      1]
 [     0      0    784    485 290667]
 ...
 [   540    483      4      2      5]
 [   535    484      1      1      1]
 [   557    484      1      1      1]] Centroides: [[286.78416699 247.87739148]
 [  0.           0.        ]
 [423.30255241 240.41489746]
 ...
 [541.8        483.8       ]
 [535.         484.        ]
 [557.         484.        ]]


In [30]:
for i in range(1, num_labels):
    x = stats[i, cv2.CC_STAT_LEFT]
    y = stats[i, cv2.CC_STAT_TOP]
    w = stats[i, cv2.CC_STAT_WIDTH]
    h = stats[i, cv2.CC_STAT_HEIGHT]
    area = stats[i, cv2.CC_STAT_AREA]
    (cX, cY) = centroids[i]
    if area > 100:
        cv2.rectangle(imagen_color_procesada, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.circle(imagen_color_procesada, (int(cX), int(cY)), 4, (0, 0, 255), -1)
cv2.imshow('ImagenEtiquetada', imagen_color_procesada)
cv2.waitKey(0)
cv2.destroyAllWindows()

Esqueleto (Silueta de la imagen) Método Manhattan
Corner,box, gaussian, bilineal, sobel

In [33]:
ruta = "img/Bailarina.png"
imagen = cv2.imread(ruta)
gris = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)

# Manhattan
bordes = np.zeros_like(gris)

for y in range(gris.shape[0]-1):
    for x in range(gris.shape[1]-1):

        dx = abs(int(gris[y, x]) - int(gris[y, x+1]))
        dy = abs(int(gris[y, x]) - int(gris[y+1, x]))

        valor = dx + dy
        bordes[y, x] = min(valor, 255)

_, bordes = cv2.threshold(bordes, 50, 255, cv2.THRESH_BINARY)

# BOX
box = cv2.blur(gris,(5,5))

# GAUSSIAN
gaussian = cv2.GaussianBlur(gris,(5,5),0)

# BILINEAR
bilineal = cv2.resize(gris,None,fx=2,fy=2,interpolation=cv2.INTER_LINEAR)

# SOBEL
sobelx = cv2.Sobel(gris,cv2.CV_64F,1,0,ksize=3)
sobely = cv2.Sobel(gris,cv2.CV_64F,0,1,ksize=3)

sobel = cv2.magnitude(sobelx,sobely)
sobel = np.uint8(np.clip(sobel,0,255))

# CORNERS
gris_float = np.float32(gris)
corners = cv2.cornerHarris(gris_float,2,3,0.04)

imagen_corners = imagen.copy()
imagen_corners[corners > 0.01 * corners.max()] = [0,0,255]

# mostrar
cv2.imshow("Original", imagen)
cv2.imshow("Manhattan", bordes)
cv2.imshow("Box", box)
cv2.imshow("Gaussian", gaussian)
cv2.imshow("Bilinear", bilineal)
cv2.imshow("Sobel", sobel)
cv2.imshow("Corners", imagen_corners)

cv2.waitKey(0)
cv2.destroyAllWindows()